In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
import os

import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numpy import linalg as LA

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')

4

In [2]:
env_seed = 7        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 200   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
# pf_res_plotly(pp_net);

In [3]:
def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret

# plot policy
def policy_plotly(policy_net, topology):
    """
    用 Plotly 绘制各母线的策略曲线，每个子图显示一个母线的 RLC-FT 策略与基线（Linear）策略比较，
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_rlc = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(5):
        baseline_vals = []
        policy_vals = []
        for j in range(N):
            # 计算基线控制值：baseline = max(state-1.05, 0) - max(0.95-state, 0)
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)  # 取负值
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[i](state_tensor, topology)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))

        baseline_vals = np.array(baseline_vals)
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        baseline_vals_scaled = 5 * baseline_vals
        
        x_vals = np.linspace(10, 14, N)
        
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            legendgroup='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=policy_vals_smoothed,
            mode='lines',
            name='RLC-FT',
            legendgroup='RLC-FT',
            showlegend=showlegend,
            line=dict(color=color_rlc)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        font=dict(size=16),
        xaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )
    
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'policy_plot.pdf')
    import plotly.io as pio
    pio.kaleido.scope.mathjax = None
    fig.write_image(output_path)
    fig.show()


def safe_net_plotly(safe_net):
    """
    用 Plotly 绘制 safe network 策略曲线，每个子图显示一个母线的 Stable-DDPG 与 Linear 比较
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_safe = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(len(safe_net)):
        baseline_vals = []
        safe_vals = []
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            # safe_net[i].get_action 接受列表输入，返回单个数值
            action = safe_net[i].get_action([state_val])
            safe_vals.append(-float(action))
        baseline_vals = np.array(baseline_vals)
        baseline_vals_scaled = 2 * baseline_vals
        x_vals = np.linspace(10, 14, N)
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=safe_vals,
            mode='lines',
            name='Safe-DDPG',
            showlegend=showlegend,
            line=dict(color=color_safe)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        xaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )

    fig.show()


def x_policy_plotly(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals_smoothed,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

In [5]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_11_a{i}.pth'))
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')
    safe_agent_net[i].load_state_dict(policy_net_dict)

In [6]:
policy_plotly(agent_policy_net, topology)

In [7]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.7* action_agent.detach().cpu().numpy()[0] #0.7
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))

2025-06-12 12:04:01.487 | SUCCESS  | __main__:<module>:48 - episode 0 stable at 18 steps
2025-06-12 12:04:02.050 | SUCCESS  | __main__:<module>:48 - episode 1 stable at 13 steps
2025-06-12 12:04:03.237 | SUCCESS  | __main__:<module>:48 - episode 2 stable at 29 steps
2025-06-12 12:04:05.247 | SUCCESS  | __main__:<module>:48 - episode 3 stable at 48 steps
2025-06-12 12:04:05.928 | SUCCESS  | __main__:<module>:48 - episode 4 stable at 14 steps
2025-06-12 12:04:07.694 | SUCCESS  | __main__:<module>:48 - episode 5 stable at 43 steps
2025-06-12 12:04:08.007 | SUCCESS  | __main__:<module>:48 - episode 6 stable at 6 steps
2025-06-12 12:04:10.524 | SUCCESS  | __main__:<module>:48 - episode 7 stable at 62 steps
2025-06-12 12:04:12.195 | SUCCESS  | __main__:<module>:48 - episode 8 stable at 42 steps
2025-06-12 12:04:12.994 | SUCCESS  | __main__:<module>:48 - episode 9 stable at 18 steps
2025-06-12 12:04:14.205 | SUCCESS  | __main__:<module>:48 - episode 10 stable at 29 steps
2025-06-12 12:04:15.7

average recovery step is:
22.52
14.271636206125772
average reactive power cost is:
81.04917605103842
127.42901727624427
the total cost is:
[284.09732508456983, 137.18514334719336, 425.9187005475086, 589.3418183524998, 190.7477823245648, 748.2545882175492, 77.38862251248439, 1195.0203166515334, 728.8824623197044, 215.85212525962586, 428.30806363033776, 564.2323358507629, 302.3783955757402, 241.04266796845556, 193.5222281738036, 163.91555974100478, 299.95673681644615, 945.1009685537468, 407.5636319080903, 538.4514144702168, 0.0, 161.70921400361073, 204.68586595278262, 308.1941207079435, 129.14888222216342, 188.6806537775513, 212.62414435767306, 378.5921394368649, 402.0160752649409, 145.6307422007521, 0.0, 432.11777042860206, 433.3880398906368, 227.10890407372705, 468.93708686230957, 371.83811080749007, 1428.3219760098755, 1131.092123280528, 53.256509321473466, 832.9185209333709, 142.06231880045647, 215.5568717490603, 160.83985489865466, 61.08081357256334, 103.925965426776, 420.6157619743

In [8]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_12_a{i}.pth'))
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')
    safe_agent_net[i].load_state_dict(policy_net_dict)

policy_plotly(agent_policy_net, topology)

In [9]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0* action_agent.detach().cpu().numpy()[0] 
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))

2025-06-12 12:08:13.371 | SUCCESS  | __main__:<module>:48 - episode 0 stable at 24 steps
2025-06-12 12:08:13.842 | SUCCESS  | __main__:<module>:48 - episode 1 stable at 10 steps
2025-06-12 12:08:14.705 | SUCCESS  | __main__:<module>:48 - episode 2 stable at 21 steps
2025-06-12 12:08:16.076 | SUCCESS  | __main__:<module>:48 - episode 3 stable at 32 steps
2025-06-12 12:08:16.593 | SUCCESS  | __main__:<module>:48 - episode 4 stable at 12 steps
2025-06-12 12:08:18.065 | SUCCESS  | __main__:<module>:48 - episode 5 stable at 36 steps
2025-06-12 12:08:19.410 | SUCCESS  | __main__:<module>:48 - episode 6 stable at 31 steps
2025-06-12 12:08:20.698 | SUCCESS  | __main__:<module>:48 - episode 7 stable at 31 steps
2025-06-12 12:08:25.569 | SUCCESS  | __main__:<module>:48 - episode 8 stable at 124 steps
2025-06-12 12:08:29.192 | SUCCESS  | __main__:<module>:48 - episode 9 stable at 86 steps
2025-06-12 12:08:30.036 | SUCCESS  | __main__:<module>:48 - episode 10 stable at 17 steps
2025-06-12 12:08:32

average recovery step is:
38.64
32.32553789188975
average reactive power cost is:
140.5893424768255
195.849190257599
the total cost is:
[437.60929479951005, 119.96071816246987, 373.19832072109597, 443.7058825898088, 210.62947167450417, 757.4527250181194, 392.3003163697716, 735.0099603524801, 2576.7146575623633, 1033.0798119913516, 328.400263142729, 704.5233523341334, 266.8768194886957, 1011.5162893543418, 147.51530618604156, 313.9669351384316, 914.8257680918871, 609.4902203875502, 756.209539750806, 384.77093614435194, 0.0, 130.16157436583333, 419.20803105534236, 207.57343863857054, 901.8450402305428, 207.50193993834716, 139.30343919160947, 1020.0307627454736, 409.2986561986554, 82.4719055318557, 268.8522903234471, 734.5130483587657, 1811.1587100355518, 788.5279549664989, 1885.2484961122552, 905.3186368058534, 988.1096140103946, 793.9643579310134, 26.91069180539673, 1761.1768607636134, 273.97104872560186, 841.7491087227094, 210.0734236984085, 441.46471992075305, 352.0577634306893, 1111.